In [0]:
from pyspark.sql.functions import col, count, when, isnan
 
# Verify dim_events
df_dim_events = spark.sql("SELECT * FROM marathos.gold.dim_events")

df_fct_results = spark.sql("SELECT * FROM marathos.gold.fct_results")

df_dim_athlete = spark.sql("SELECT * FROM marathos.gold.dim_athlete")

# count unique events
print("number of unique events:", df_dim_events.count())

# check duplicates
print("number of duplicates on event_id:", df_dim_events.count() - df_dim_events.dropDuplicates(["event_id"]).count())

print("number of duplicates on fct_results:", df_fct_results.count() - df_fct_results.dropDuplicates(["result_id"]).count())

print("number of duplicates on dim_athlete:", df_dim_athlete.count() - df_dim_athlete.dropDuplicates(["athlete_id"]).count())

# Verify miles conversion to km 
print("\nMiles events sample:")
df_dim_events.filter(col("event_unit_type") == "mi").select(
    "event_name", "event_distance_value", "event_distance_km"
).show(5)

# Validate null values in key columns
print("\nNull-values per column in dim_events:")
df_dim_events.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in ["event_id","event_name", "event_unit_type", "host_country_code"]
]).show()

print("\nNull-values per column in fct_results:")
df_fct_results.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in ["event_id","result_id", "athlete_id"]
]).show()

In [0]:
%sql
-- verify marts_events_calendar
SELECT *
FROM marathos.gold.mart_events_calendar
LIMIT 20

In [0]:
%sql
 -- I verify if the event_id is unique for each year
SELECT
    event_name,
    year_of_event,
    event_id,
    count(*) as antal
    FROM marathos.gold.dim_events
    WHERE event_name = 'Ultravasan90'
    GROUP BY event_name, year_of_event, event_id
    ORDER BY year_of_event

In [0]:
%sql
-- is it possible for same athlete to run same event same year
SELECT event_id, athlete_id, count(*) as amount
FROM marathos.gold.fct_results
GROUP BY event_id, athlete_id
HAVING amount > 1
ORDER BY amount DESC
LIMIT 10

In [0]:
%sql
-- Verify no duplicate rows exist for same athlete, event and performance
-- WHY: ensures result_id hash in fct_results is unique per row

SELECT event_id, athlete_id, athlete_performance, count(*) as amount
FROM marathos.silver.marathon_results
GROUP BY event_id, athlete_id, athlete_performance
HAVING amount > 1
ORDER BY amount DESC
LIMIT 10

I verify result_id uniqueness in fct_results. 
I need to ensures SHA256 hash produces unique result_id per row.  
I Expect 0 rows returned - each result_id should be unique
If rows are returned, it indicates hash collision or duplicate data

In [0]:
%sql
SELECT result_id, count(*) AS number_of_events
FROM marathos.gold.fct_results
GROUP BY result_id
HAVING count(*) > 1
LIMIT 10

Here i verify that dim_events has same unique events per year for silver and gold layer. 

Silver contains one row per athltete, meaning same events appear multiple times. Therefore distinct is used.  

Gold dim_events contains one row per unique event due to dropDuplicates, so count(*) is sufficient.


In [0]:
%sql
-- Number of unique events per year in Silver layer
SELECT year_of_event, count(distinct event_id) as number_of_events
FROM marathos.silver.marathon_results
GROUP BY year_of_event
ORDER BY year_of_event DESC
LIMIT 10

In [0]:
%sql
-- Number of events in Gold layer
SELECT year_of_event, COUNT(DISTINCT event_id) AS number_of_events_gold
FROM marathos.gold.dim_events
GROUP BY year_of_event
ORDER BY year_of_event DESC
LIMIT 10

Same result for silver and layer, dim_events is correct populated

I want to verify that my mart for swedish_races is same  as silver. its highest priority for my stakholders.

In [0]:
%sql
SELECT COUNT(DISTINCT event_name) AS unique_swedish_events
FROM marathos.silver.marathon_results
WHERE host_country_code = 'SWE'


In [0]:
%sql
SELECT COUNT(DISTINCT event_name) as count_swe_events
FROM marathos.gold.mart_swedish_races

Same number of events, my swedish_mart is alligned with silver.

In [0]:
%sql
SELECT * 
FROM marathos.silver.marathon_results
LIMIT 10

I validate after refactoring to have a DRY code

In [0]:
%sql
SELECT COUNT(*)
FROM marathos.silver.marathon_results

In [0]:
df = spark.sql("SELECT * FROM marathos.bronze.raw_ultra_marathons")
print("After bronze read:", df.count())

In [0]:
%sql
SELECT count(*) as antal
FROM marathos.bronze.raw_ultra_marathons

In [0]:
%sql
SELECT count(*) as antal
FROM marathos.silver.marathon_results

In [0]:
%sql
SELECT count(*) as antal
FROM marathos.gold.fct_results

In [0]:
%sql
SELECT *
FROM marathos.bronze.raw_ultra_marathons
WHERE `Event name` = 'Marathos Iberia Ultra (ESP)'
LIMIT 10

In [0]:
%sql
SELECT * 
FROM marathos.silver.marathon_results
WHERE event_name = 'Marathos Iberia Ultra'
LIMIT 10

In [0]:
%sql
SELECT *
FROM marathos.gold.dim_events
WHERE event_name = 'Marathos Iberia Ultra'


In [0]:
%sql
SELECT * 
FROM marathos.gold.dim_country


In [0]:
%sql
SELECT * 
FROM marathos.gold.dim_date


In [0]:
%sql
SELECT COUNT(*) 
FROM marathos.silver.marathon_results
WHERE host_country_code = 'SWE';


In [0]:
%sql
select count(*) as antal 
from marathos.gold.dim_events
where host_country_code = 'SWE'

In [0]:
%sql
SELECT DISTINCT event_name
FROM marathos.gold.mart_swedish_races
WHERE event_name NOT IN (
    SELECT DISTINCT event_name
    FROM marathos.gold.mart_swedish_races
    WHERE year_of_event < 2024
)
ORDER BY event_name

In [0]:
%sql
SELECT year_of_event, COUNT(DISTINCT event_name) AS antal
FROM marathos.gold.mart_swedish_races
GROUP BY year_of_event
ORDER BY year_of_event DESC
LIMIT 10

In [0]:
%sql
SELECT COUNT(DISTINCT event_name) AS unique_events
FROM marathos.silver.marathon_results
WHERE host_country_code = 'SWE'

In [0]:
%sql
SELECT count(DISTINCT event_name) AS antal
FROM marathos.gold.dim_events
WHERE host_country_code = 'SWE'